# EdgeRAG — the original research prototype

**This notebook is kept for provenance, not for use.** It is the exploratory work that
EdgeRAG grew out of, cleaned up so that it actually runs. The production system lives in
`backend/edgerag/` and is what you should build on.

Three things have been changed from the original:

1. **Two syntax errors are fixed.** As saved, the original could not be executed: a `print`
   was missing a closing parenthesis, and `def hybrid_search(query, k=):` had an empty
   default value.
2. **The reranker is now initialized explicitly.** The original called `reranker.predict(...)`
   against a name that was never assigned in any cell — it only worked because of hidden
   kernel state from a session that is not in the file.
3. **The corpus is now yours.** The original ran against a copyrighted textbook, which is
   deliberately not in this repository. Point `PDF_PATH` at any PDF you own, or at
   `../samples/retrieval-primer.pdf`.

Everything else is preserved, including the design decisions that turned out to be wrong.
Those are called out where they appear — they are the reason EdgeRAG exists.

---

### Known limitations of this prototype

| Problem | Consequence | Fixed in EdgeRAG by |
| --- | --- | --- |
| Fusion is concatenate + string-dedupe | A sparse-only result can never outrank a dense one | `rag/fusion.py` — reciprocal rank fusion |
| `bm25_retriever.k = 5` against a dense `k=40` | The keyword arm is nearly irrelevant | Matched budgets, both configurable |
| Chroma built in memory | The whole index is rebuilt on every restart | Persistent NumPy and Chroma stores |
| Confidence check only prints a warning | Answers confidently on a rerank score of −1.58 | `rag/confidence.py` — four signals, real abstention |
| `Source N` labels with no mapping back | Citations cannot be verified | Real chunk IDs, page numbers, validated markers |
| `<think>` blocks in the output | Chain of thought leaks into answers | `rag/citations.py` — `strip_reasoning` |
| Hardcoded persona, paths, models | Only works for one book | `core/config.py` |


## Setup

Measured on the original run: CPU only, `torch 2.11.0+cpu`, `cuda.is_available() == False`.
Every timing below came from that machine.


In [ ]:
%pip install -q langchain-community langchain-text-splitters langchain-huggingface \
               pymupdf sentence-transformers rank_bm25 chromadb ollama


In [ ]:
import sys, time
import torch

print(sys.executable)
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())


## 1. Ingestion

Original measurement: **607 pages → 3,200 chunks in 6.13 s**.

EdgeRAG additionally records each chunk's character offsets in the source document, which is
what lets a citation open the viewer at the exact passage rather than merely the right page.


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Point this at a document you own. The original textbook is not in this repository.
PDF_PATH = '../samples/retrieval-primer.pdf'

start = time.time()
pages = PyMuPDFLoader(PDF_PATH).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(pages)

# FIXED: the original was missing a closing parenthesis here and would not parse.
print(f'Loaded {len(pages)} pages -> {len(chunks)} chunks in {time.time() - start:.2f}s')


## 2. Dense index

Original measurement: **416.58 s** for 3,200 chunks on CPU.

> The mistake worth noticing: no `persist_directory` is passed, so this Chroma client is
> in-memory. That 417 seconds was paid again on every kernel restart. EdgeRAG persists both
> indexes to disk, so a restart costs nothing.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

start = time.time()
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='research_collection',
    # persist_directory='./chroma'  # <- absent in the original; this is the bug
)
print(f'Vector store built in {time.time() - start:.2f}s')


## 3. Sparse index

Original measurement: **1.49 s**. Two hundred and eighty times faster than the dense index,
and it is the only thing that reliably finds exact strings — identifiers, section numbers,
error codes.

> The `k = 5` below is the second bug. Dense retrieval fetches 40 candidates; BM25 fetches 5.
> The keyword half of a 'hybrid' system was contributing almost nothing.


In [ ]:
from langchain_community.retrievers import BM25Retriever

start = time.time()
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5   # <- should match the dense budget
print(f'BM25 index built in {time.time() - start:.2f}s')


## 4. Fusion

> The third bug, and the most consequential. Concatenating `dense + sparse` and dropping exact
> string duplicates means **order is inherited from the dense list**. A chunk that BM25 ranked
> first can never outrank a chunk dense ranked first, no matter how relevant it is, and
> agreement between the two retrievers earns nothing at all.
>
> EdgeRAG replaces this with reciprocal rank fusion, where a chunk found by both retrievers is
> promoted above one found by only one. See `backend/edgerag/rag/fusion.py`.


In [ ]:
def hybrid_search(query, k=40):   # FIXED: `k=` in the original was a syntax error
    dense_results = vector_db.similarity_search(query, k=k)
    sparse_results = bm25_retriever.invoke(query)[:k]

    unique, seen = [], set()
    for doc in dense_results + sparse_results:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique.append(doc)
    return unique


## 5. Reranking

Original measurement: **0.35–0.42 s for 40 candidates**.

> The original notebook called `reranker.predict(...)` without ever assigning `reranker` in any
> cell. It ran only because of hidden kernel state. The initialization below is the fix, and it
> is why EdgeRAG injects providers explicitly instead of relying on globals.

Cross-encoder scores are raw logits, roughly −10 to +10. They are **not** probabilities and are
not comparable across models.


In [ ]:
from sentence_transformers import CrossEncoder

# ADDED: never present in the original.
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def reranked_hybrid_search(query, top_n=40, final_k=5):
    candidates = hybrid_search(query, k=top_n)
    if not candidates:
        return []

    start = time.time()
    scores = reranker.predict([[query, doc.page_content] for doc in candidates])
    elapsed = time.time() - start

    for doc, score in zip(candidates, scores):
        doc.metadata['rerank_score'] = float(score)

    ranked = sorted(candidates, key=lambda d: d.metadata['rerank_score'], reverse=True)
    print(f'Reranked {len(candidates)} candidates in {elapsed:.2f}s')
    for i, doc in enumerate(ranked[:final_k]):
        page = doc.metadata.get('page', '?')
        print(f'  {i + 1}. score={doc.metadata["rerank_score"]:+.4f}  page={page}')
    return ranked[:final_k]


## 6. Generation

Original measurement: **19.42–24.61 s** with `deepseek-r1:1.5b` on CPU.

> Three problems live in this cell.
>
> **The confidence check does nothing.** It prints a warning and answers anyway. On one recorded
> run the top reranker score was **−1.5837** — strongly negative — and the notebook produced a
> confident answer regardless. EdgeRAG blends four signals and actually abstains.
>
> **`Source N` is not a citation.** The labels have no mapping back to a document or page, so
> nothing in the answer can be verified. EdgeRAG returns real chunk IDs and validates every
> marker the model emits against the context it was actually given.
>
> **`<think>` blocks leak.** Visible in the original outputs. EdgeRAG strips them, including
> mid-stream.


In [ ]:
import ollama   # the original never executed this import

def ask(query, top_n=40, final_k=5):
    context_chunks = reranked_hybrid_search(query, top_n=top_n, final_k=final_k)
    if not context_chunks:
        return 'Nothing was retrieved.'

    top_score = context_chunks[0].metadata.get('rerank_score', 0.0)
    if top_score < 0:
        # The prototype only warned here. EdgeRAG stops.
        print(f'Low confidence (top score {top_score:+.4f}) - the answer below is unreliable.')

    context = '\n\n'.join(
        f'Source {i + 1} (page {doc.metadata.get("page", "?")}):\n{doc.page_content}'
        for i, doc in enumerate(context_chunks)
    )

    system = (
        'Answer only from the provided sources. Cite the source number for every claim. '
        'If the sources do not contain the answer, say so explicitly.'
    )

    start = time.time()
    response = ollama.chat(
        model='deepseek-r1:1.5b',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {query}'},
        ],
    )
    answer = response['message']['content']

    # The original displayed this verbatim, <think> blocks included.
    import re
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()

    print(f'\nGenerated in {time.time() - start:.2f}s\n')
    print(answer)
    return answer


In [ ]:
ask('What is the definition of a project?')


---

## Where this went

Everything above is a single linear script with global state, one hardcoded corpus, and no way
to inspect why a particular answer came out the way it did. EdgeRAG keeps the ideas — hybrid
retrieval, cross-encoder reranking, local generation — and adds the parts that make them
trustworthy:

- rank fusion that rewards retriever agreement instead of inheriting dense order,
- persistent indexes, so the 417 seconds is paid once,
- citations that resolve to a document, a page and a character span,
- abstention that actually abstains,
- per-stage telemetry surfaced in the interface,
- provider interfaces, so none of this is welded to Chroma, Ollama or LangChain.

```bash
pip install -e 'backend[all]'
edgerag doctor
edgerag serve
```
